# Parameter estimation

_Probability & Statistics_

**Guess a distribution's hidden numbers from data. And know when your guess is fair.**

The real world doesn't hand you a distribution's true mean or variance.

     You only have data. So you estimate those hidden numbers from samples.

---

This notebook builds parameter estimation one step at a time — estimating a normal's mean and variance from a sample, comparing the biased and unbiased variance formulas, and checking that the estimators are fair on average. Run each cell and read the note above it as you go. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

### Step 1 — Draw a sample from a known normal

We pick a *true* mean and standard deviation, then draw 2000 samples from `Normal(5, 2²)`. Because we set the truth ourselves, we can later check how close our estimates come. In the real world you'd only have the sample `x` and would never see these true values.

In [ ]:
from scipy.stats import norm

rng = np.random.default_rng(0)
true_mu, true_sigma = 5.0, 2.0

x = rng.normal(true_mu, true_sigma, size=2000)

### Step 2 — Point estimates: mean and two variances

The sample mean estimates `mu`. For variance there are two formulas: dividing by `n` (`ddof=0`) gives the **biased** estimate that runs slightly low, while dividing by `n-1` (`ddof=1`) gives the **unbiased** estimate. `scipy`'s maximum-likelihood `norm.fit` recovers the same mean and the `n`-divisor (biased) standard deviation.

In [ ]:
# Point estimates.
mu_hat = x.mean()
var_biased = x.var(ddof=0)        # divides by n   -> biased low
var_unbiased = x.var(ddof=1)      # divides by n-1 -> unbiased
print("mu_hat      =", round(mu_hat, 4), " (true 5)")
print("var ddof=0  =", round(var_biased, 4))
print("var ddof=1  =", round(var_unbiased, 4), " (true 4)")

# scipy MLE fit recovers the same parameters.
fmu, fsigma = norm.fit(x)
print("norm.fit mu =", round(fmu, 4), " sigma =", round(fsigma, 4))

### Step 3 — Confirm the mean estimator is unbiased

A single sample's `mu_hat` won't equal the truth exactly. **Unbiasedness** is a statement about the *average* over many independent datasets. We draw 5000 fresh samples, take each one's mean, and average those — the result should sit right on the true mean of 5.

In [ ]:
# Unbiasedness: average mu_hat over many datasets equals the truth.
ests = [rng.normal(true_mu, true_sigma, 100).mean() for _ in range(5000)]
print("mean of mu_hat over datasets =", round(np.mean(ests), 4))

## Visualize the data & results

_Do sample estimators recover a normal's true mean and variance?_

### Step 1 — Re-estimate from a fresh sample

For the plot we draw a new `Normal(5, 2²)` sample and compute the same three estimates: the sample mean and both variance formulas. We collect them next to the true values (`mu = 5`, `var = 4`) so the bars can be compared directly.

In [ ]:
# Estimate mu and variance from a Normal(5, sigma^2=4) sample.
rng = np.random.default_rng(0)
x = rng.normal(5.0, 2.0, size=2000)

mu_hat = x.mean()
var0 = x.var(ddof=0)
var1 = x.var(ddof=1)

vals = [mu_hat, 5.0, var0, var1, 4.0]

### Step 2 — Bar chart: estimates vs truth

Each estimate sits beside its true value. The mean bar should nearly match `true mu`, and both variance bars should land close to `true var = 4` — with `ddof=0` a touch lower than `ddof=1`. At `n=2000` the bias is tiny, which is exactly why it's easy to overlook.

In [ ]:
labels = ['mu_hat', 'true mu', 'var (ddof=0)', 'var (ddof=1)', 'true var']
colors = ['#4ea1ff', '#7ee787', '#c89bff', '#c89bff', '#7ee787']
plt.bar(labels, vals, color=colors)
plt.title('Estimates vs truth for Normal(mu=5, sigma^2=4), n=2000')
plt.ylabel('value')
plt.show()